In [2]:
%cd ..

/home/oleg/audio-llm-yandex-camp


In [49]:
import re
from pathlib import Path
import json
from src.asr_eval.align.parsing import parse_multivariant_string
from src.asr_eval.align.data import MultiVariant
import numpy as np

In [22]:
words_text = Path('datasets/tachki/тачки_слова.txt').read_text()

In [30]:
word_to_multivariant = {}

for block in words_text.splitlines()[0].split(', '):
    assert block.startswith('{')
    assert block.endswith('}')
    for option in block[1:-1].split('|'):
        assert option not in word_to_multivariant
        word_to_multivariant[option] = block

word_to_multivariant

{'джили': '{джили|geeley}',
 'geeley': '{джили|geeley}',
 'тойота': '{тойота|тёха|toyota}',
 'тёха': '{тойота|тёха|toyota}',
 'toyota': '{тойота|тёха|toyota}',
 'хендай': '{хендай|хёндай|hyundai}',
 'хёндай': '{хендай|хёндай|hyundai}',
 'hyundai': '{хендай|хёндай|hyundai}',
 'киа': '{киа|kia}',
 'kia': '{киа|kia}',
 'форд': '{форд|фурда|ford}',
 'фурда': '{форд|фурда|ford}',
 'ford': '{форд|фурда|ford}',
 'хонда': '{хонда|honda}',
 'honda': '{хонда|honda}',
 'ниссан': '{ниссан|нисан|nissan}',
 'нисан': '{ниссан|нисан|nissan}',
 'nissan': '{ниссан|нисан|nissan}',
 'мерседес': '{мерседес|мерс|мерин|mercedes}',
 'мерс': '{мерседес|мерс|мерин|mercedes}',
 'мерин': '{мерседес|мерс|мерин|mercedes}',
 'mercedes': '{мерседес|мерс|мерин|mercedes}',
 'бмв': '{бмв|бэха|бумер|bmw}',
 'бэха': '{бмв|бэха|бумер|bmw}',
 'бумер': '{бмв|бэха|бумер|bmw}',
 'bmw': '{бмв|бэха|бумер|bmw}',
 'ауди': '{ауди|audi}',
 'audi': '{ауди|audi}',
 'фольксваген': '{фольксваген|фольц|жук|volkswagen}',
 'фольц': '{фольк

In [23]:
words = set([
    word.lower() for word in re.findall(r'\w+', words_text) if not re.search(r'[0-9]', word) and len(word) > 2
])

', '.join(sorted(words))

'audi, baic, bmw, brilliance, buick, cadillac, chana, chery, chevrolet, citroen, citroën, dodge, ford, foton, gaz, geeley, geely, haval, honda, hyundai, infiniti, jac, jaguar, jeep, jmc, kia, lada, land, lexus, lifan, lincoln, mazda, mercedes, mini, mitsubishi, moskvich, nissan, opel, peugeot, porsche, renault, reno, rover, saab, skoda, smart, sollers, subaru, suzuki, tagaz, toyota, uaz, volkswagen, volvo, zotye, абс, авилон, авто, автоваз, автоград, автоимпорт, автомир, автопорт, автосалоны, автоспеццентр, автоцентр, аделя, адреса, академика, алтайавтоцентр, арена, арктика, архангельск, астрахань, ауди, байк, бакра, балтавтотрейд, балтия, барнаул, башавтоком, белая, белгород, белогорье, беляево, бенц, бмв, боевая, боровское, бриллианс, брэнд, буик, бумер, бутово, буханка, бьюик, бэха, ваз, ватутина, витебский, владивосток, волга, волгина, вольво, воронеж, восток, газ, газик, генерала, горький, дача, джак, джей, джили, джип, динамика, додж, доджик, дом, домостроительная, донская, дону,

In [ ]:
text = Path('datasets/tachki/тачки_тексты.txt').read_text()

blocks = [' '.join(l.strip().lower() for l in block.splitlines()) for block in text.split('\n\n')]
for i, block in enumerate(blocks):
    items = list(word_to_multivariant.items())
    for i2, (k, v) in enumerate(items):
        block = block.replace(k, f'<<{i2}>>')
    for i2 in range(len(items)):
        block = block.replace(f'<<{i2}>>', items[i2][1])
    blocks[i] = block
    print(block)

In [53]:
def is_cyrillic(text: str):
    for char in text:
        if not (0x0400 <= ord(char) <= 0x04FF) and char != ' ':
            return False
    return True

lines = Path('datasets/tachki/multivariant.txt').read_text().splitlines()
for line in lines:
    tokens = parse_multivariant_string(line)
    for t in tokens[::-1]:
        if isinstance(t, MultiVariant):
            start, end = t.pos
            orig_options = [' '.join(str(t.value) for t in option) for option in t.options]
            options = [x for x in orig_options if is_cyrillic(x)]
            option = np.random.choice(options)
            line = line[:start] + option + line[end:]
    line = line.replace(' .', '.').replace(' ,', ',').replace(' !', '!').replace(' ?', '?')
    print(line)

добрый день, это рольф хендай на ленинградском проспекте? да, слушаю вас. чем могу помочь? интересует новый tucson. говорят, у вас сейчас скидки? да, акция до конца месяца: скидка или бесплатное то. а хендай в кредит можно? процентовка какая? можете подъехать на тест драйв?
смотри, бумер новая! да, но мне больше бэха нравится. добрый день! хотите тест драйв? а сколько мерседес в trade in дадите за мою копейку? оценим. но лучше сразу кредит у нас выгодные программы.
дорогой, киа sportage вроде надежный? да, но киа дорогая в обслуживании. может, шкоду? консультант сказал, у киа гарантия. ладно, давай возьмём, но в кредит чтоб не всю сумму сразу.
пап, смотри, ведро новое! гранта спорт. ты серьёзно? за эти деньги лучше подержанного немца. но тут же гарантия! и печка нормальная! ладно, давай посчитаем, сколько переплатим в кредит.
хочу в лизинг. какие условия? а если у меня плохая кредитная история? тогда нужен поручитель или залог. и процент выше.
офигеть, порш! мечта. да брось, за эти ден